In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Import matplotlib only for displaying sample images (keeps the rest of the pipeline unchanged)
import matplotlib.pyplot as plt


# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
    transforms.Resize((28, 28)),
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor
     transforms.ToTensor(),
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
      transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here

batch_size = 64  # Reasonable default batch size for EMNIST + transfer learning

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,        # Shuffle training data
    num_workers=2,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,       # No shuffle for evaluation
    num_workers=2,
)

images, labels = next(iter(train_loader))

# Unnormalize helper (ImageNet stats used above)
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

# Move mean/std to same device as images (in case DataLoader returns pinned/GPU tensors later)
mean = mean.to(images.device)
std = std.to(images.device)

# Unnormalize and clamp to valid display range
images_vis = images * std + mean
images_vis = images_vis.clamp(0, 1)

# Plot a small grid of samples
num_show = 12  # Number of images to display
cols = 6
rows = (num_show + cols - 1) // cols

plt.figure(figsize=(12, 4))
for i in range(num_show):
    ax = plt.subplot(rows, cols, i + 1)
    img = images_vis[i].permute(1, 2, 0).cpu().numpy()  # CHW -> HWC for matplotlib
    ax.imshow(img)
    # EMNIST letters labels are 1..26, map to 0..25 for indexing into 'letters'
    label_idx = int(labels[i].item()) - 1
    ax.set_title(letters[label_idx] if 0 <= label_idx < 26 else str(labels[i].item()))
    ax.axis("off")

plt.tight_layout()
plt.show()
##done with part 1 (hopefully?)





In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

# 1) Select device for model placement (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2) Load pretrained EfficientNetV2-Small with ImageNet weights
#    - Using the official weights enum ensures correct pretrained checkpoint + metadata
model = efficientnet_v2_s(pretrained=True)


# 3) Freeze the backbone so only the classifier head trains
for param in model.features.parameters():
    param.requires_grad = False  # Disable gradients for backbone parameters


# 4) Replace the classifier head to output 26 classes
in_features = model.classifier[1].in_features  # Read input size to the final linear layer
model.classifier[1] = nn.Linear(in_features, num_classes)  # Replace with 26-class output layer

# 5) Ensure the new classifier parameters are trainable
for param in model.classifier.parameters():
    param.requires_grad = True

# Move model to device
model = model.to(device)

# (Optional sanity checks) Print model and number of trainable parameters
print(model)  # Show architecture with updated classifier

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)  # Count trainable params
total_params = sum(p.numel() for p in model.parameters())  # Count total params
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")  # Confirm only head is trainable


In [ ]:
# Write your code here


import torch
import torch.nn as nn
from tqdm import tqdm


def train_one_epoch(model, dataloader, criterion, optimizer, device):

    model.train()  # Enable training mode (dropout/batchnorm behave accordingly)

    total_loss = 0.0  # Accumulate batch losses to compute epoch average
    correct = 0  # Count correct predictions for accuracy
    total = 0  # Count total samples seen

    for images, labels in tqdm(dataloader):  # Iterate with a progress bar
        # Move batch to the selected device (GPU)
        images = images.to(device)
        labels = labels.to(device)

        # Convert EMNIST labels for compatibility
        labels = labels - 1

        # Forward pass: get raw logits of shape [batch_size, 26]
        outputs = model(images)

        # Compute classification loss (CrossEntropy expects logits + class indices)
        loss = criterion(outputs, labels)

        # Standard optimization step
        optimizer.zero_grad()  # Clear old gradients
        loss.backward()  # Backpropagate
        optimizer.step()  # Update trainable parameters

        # Track loss
        total_loss += loss.item()

        # Track accuracy
        predictions = torch.softmax(outputs, dim=1)  # Convert logits to probabilities
        predictions = torch.argmax(predictions, dim=1)  # Pick the most likely class index
        correct += (predictions == labels).sum().item()  # Count correct predictions
        total += labels.size(0)  # Update total samples

    # Compute epoch-level metrics
    avg_loss = total_loss / len(dataloader)  # Average loss per batch
    accuracy = 100.0 * correct / total  # Accuracy percentage

    return avg_loss, accuracy  # Return metrics for logging/plotting


def validate(model, dataloader, criterion, device):

    model.eval()  # Set evaluation mode (disables dropout, uses running stats for batchnorm)

    total_loss = 0.0  # Accumulate batch losses
    correct = 0  # Count correct predictions
    total = 0  # Count total samples

    with torch.no_grad():  # Disable gradient computation for faster/cheaper evaluation
        for images, labels in dataloader:
            # Move batch to device
            images = images.to(device)
            labels = labels.to(device)

            # Convert EMNIST labels from 1..26 to 0..25
            labels = labels - 1

            # Forward pass
            outputs = model(images)

            # Compute loss
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Compute accuracy
            predictions = torch.softmax(outputs, dim=1)
            predictions = torch.argmax(predictions, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    # Compute epoch-level metrics
    avg_loss = total_loss / len(dataloader)
    accuracy = 100.0 * correct / total

    return avg_loss, accuracy  # Return metrics for logging/plotting

In [ ]:
# Write your code here
import torch
import torch.nn as nn
import torch.optim as optim  # Import optimizers (s Adam/AdamW (rememeber stage 2))
import matplotlib.pyplot as plt



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Use GPU if available

# 2) Move model to device
model = model.to(device)  # Keep model on the correct device for training/evaluation

# 3) Define loss function
criterion = nn.CrossEntropyLoss()  # Works with logits and integer class indices (0..25)

# 4) Define optimizer
#    Since i froze model.features, only classifier parameters have requires_grad=True.
#    use Adam with a small learning rate.
optimizer = optim.Adam(model.parameters(), lr=0.0001)  # Optimizes only trainable params (classifier head)

# 5) Training configuration
num_epochs = 10  # Reasonable number of epochs for transfer learning on EMNIST

# 6) Lists to store metrics for plotting
train_losses = []  # Store average training loss per epoch
val_losses = []  # Store average validation loss per epoch
train_accuracies = []  # Store training accuracy per epoch
val_accuracies = []  # Store validation accuracy per epoch

# 7) Training loop over epochs (uses the train_one_epoch/validate functions defined in Part 3)
for epoch in range(num_epochs):
    # Train for one epoch and get metrics
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)

    # Validate on the test set and get metrics
    val_loss, val_acc = validate(model, test_loader, criterion, device)

    # Store metrics for plotting
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    # Print epoch summary
    print(
        f"Epoch {epoch+1}/{num_epochs}: "
        f"Train Loss={train_loss:.4f}, Train Accuracy={train_acc:.2f}%, "
        f"Val Loss={val_loss:.4f}, Val Accuracy={val_acc:.2f}%"
    )

# 8) Plot Loss & Accuracy curves
plt.figure(figsize=(12, 5))  # Create a wide figure with two subplots

# Plot loss curve
plt.subplot(1, 2, 1)  # First subplot for loss
plt.plot(range(1, num_epochs + 1), train_losses, label="Train Loss", marker="o")  # Training loss
plt.plot(range(1, num_epochs + 1), val_losses, label="Validation Loss", marker="o")  # Validation loss
plt.xlabel("Epochs")  # X-axis label
plt.ylabel("Loss")  # Y-axis label
plt.title("Loss Curve")  # Plot title
plt.legend()  # Show legend

# Plot accuracy curve
plt.subplot(1, 2, 2)  # Second subplot for accuracy
plt.plot(range(1, num_epochs + 1), train_accuracies, label="Train Accuracy", marker="o")  # Training accuracy
plt.plot(range(1, num_epochs + 1), val_accuracies, label="Validation Accuracy", marker="o")  # Validation accuracy
plt.xlabel("Epochs")  # X-axis label
plt.ylabel("Accuracy (%)")  # Y-axis label
plt.title("Accuracy Curve")  # Plot title
plt.legend()  # Show legend

plt.tight_layout()  # Prevent subplot overlap
plt.show()  # Render the plots

In [ ]:
# Write your code here

def validate_tta(model, dataloader, criterion, device):

    model.eval()  # Set evaluation mode (disables dropout, uses running stats for batchnorm)

    total_loss = 0.0  # Accumulate batch losses to compute epoch average
    correct = 0  # Count correct predictions
    total = 0  # Count total samples

    with torch.no_grad():  # Disable gradients for evaluation
        for images, labels in dataloader:
            # Move batch to device
            images = images.to(device)
            labels = labels.to(device)

            # Convert EMNIST labels from 1..26 to 0..25 (CrossEntropyLoss expects 0-indexed classes)
            labels = labels - 1

            # 1) Forward pass on original images -> logits [B, 26]
            logits_orig = model(images)

            # 2) Forward pass on horizontally flipped images (flip width dimension)
            h_flipped = torch.flip(images, dims=[3])  # dims=[3] corresponds to W in NCHW
            logits_h = model(h_flipped)

            # 3) Forward pass on vertically flipped images (flip height dimension)
            v_flipped = torch.flip(images, dims=[2])  # dims=[2] corresponds to H in NCHW
            logits_v = model(v_flipped)

            # Average the 3 logits (logit averaging is standard for TTA ensembles)
            logits_avg = (logits_orig + logits_h + logits_v) / 3.0

            # Compute loss using averaged logits
            loss = criterion(logits_avg, labels)
            total_loss += loss.item()

            # Compute accuracy using averaged logits
            predictions = torch.softmax(logits_avg, dim=1)
            predictions = torch.argmax(predictions, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    # Compute epoch-level metrics
    avg_loss = total_loss / len(dataloader)
    accuracy = 100.0 * correct / total

    return avg_loss, accuracy  # Return metrics for logging/plotting



tta_loss, tta_acc = validate_tta(model, test_loader, criterion, device)  # Run TTA validation on test set
print(f"TTA Validation: Loss={tta_loss:.4f}, Accuracy={tta_acc:.2f}%")  #pribnt
